# Caso 1

In [44]:
# Funciones auxiliares — Caso 1: Árbol de regresión (Car Price)
# Son las mismas que usamos en el laboratorio de árboles (caso MPG).
# Ejecuta esta celda una vez; después sólo llama a las funciones.
#   evaluar_regresion ahora también reporta el MAPE (%).
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics import (mean_absolute_error, mean_absolute_percentage_error,
                             mean_squared_error, r2_score)


def evaluar_regresion(nombre, y_true, y_pred):
    """Imprime y regresa R², RMSE, MAE y MAPE (%) de un modelo de regresión."""
    metricas = {
        'modelo': nombre,
        'R²': round(r2_score(y_true, y_pred), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_true, y_pred)), 4),
        'MAE': round(mean_absolute_error(y_true, y_pred), 3),
        'MAPE (%)': round(mean_absolute_percentage_error(y_true, y_pred) * 100, 2),
    }
    print(f"{nombre}: R² {metricas['R²']} | RMSE {metricas['RMSE']} | "
          f"MAE {metricas['MAE']} | MAPE {metricas['MAPE (%)']}%")
    return metricas


def plot_importancias(modelo, columnas, top_n=None):
    """Importancia de cada variable en un modelo basado en árboles."""
    imp = pd.Series(modelo.feature_importances_, index=columnas).sort_values(ascending=True)
    if top_n:
        imp = imp.tail(top_n)

    fig = px.bar(
        x=imp.values, y=imp.index, orientation='h', text_auto='.3f',
        labels={'x': 'Importancia (reducción de impureza)', 'y': ''},
        title='Importancia de variables', template='plotly_white'
    )
    fig.update_layout(width=800, height=450)
    fig.show()
    return imp.sort_values(ascending=False)



In [45]:
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/jorgermzg15/data-files/refs/heads/main/CarPrice_dataset.csv')

# Mostrar las primeras filas
print("Muestra del dataset:")
df.head()

Muestra del dataset:


,CarName,carbody,fueltype,enginetype,cylindernumber,doornumber,symboling,wheelbase,carlength,carwidth,...,curbweight,enginesize,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,alfa-romero giulia,convertible,gas,dohc,4,2,3,88.6,168.8,64.1,...,2548,130,3.47,2.68,9.0,111,5000,21,27,13495.0
1,alfa-romero stelvio,convertible,gas,dohc,4,2,3,88.6,168.8,64.1,...,2548,130,3.47,2.68,9.0,111,5000,21,27,16500.0
2,alfa-romero Quadrifoglio,hatchback,gas,ohcv,6,2,1,94.5,171.2,65.5,...,2823,152,2.68,3.47,9.0,154,5000,19,26,16500.0
3,audi 100 ls,sedan,gas,ohc,4,4,2,99.8,176.6,66.2,...,2337,109,3.19,3.40,10.0,102,5500,24,30,13950.0
4,audi 100ls,sedan,gas,ohc,5,4,2,99.4,176.6,66.4,...,2824,136,3.19,3.40,8.0,115,5500,18,22,17450.0


In [46]:
from sklearn.model_selection import train_test_split

# Codificar las variables categóricas y separar el precio como variable objetivo.
X = pd.get_dummies(df.drop(columns='price'), drop_first=True)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

In [47]:
from sklearn.tree import DecisionTreeRegressor

modelo_arbol = DecisionTreeRegressor(random_state=0)
modelo_arbol.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",0
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with 

In [48]:
importancias_variables = plot_importancias(modelo_arbol, X_train.columns)
importancias_variables

enginesize                              0.674775
curbweight                              0.226543
carwidth                                0.017586
compressionratio                        0.015022
horsepower                              0.011427
                                          ...   
CarName_volvo 246                       0.000000
CarName_vokswagen rabbit                0.000000
CarName_volvo 144ea                     0.000000
CarName_volvo 264gl                     0.000000
CarName_volkswagen 1131 deluxe sedan    0.000000
Length: 173, dtype: float64

In [49]:
variable_principal = importancias_variables.idxmax()
importancia_principal = importancias_variables.max()

print(f"{variable_principal}: {importancia_principal:.4f}")

enginesize: 0.6748


In [50]:
importancias_ordenadas = importancias_variables.sort_values(ascending=False)
importancia_acumulada = importancias_ordenadas.cumsum()
variables_80 = importancias_ordenadas.loc[
    importancia_acumulada.shift(fill_value=0) < 0.80
]

resultado_80 = variables_80.rename_axis('variable').reset_index(name='importancia')
resultado_80['importancia_acumulada'] = resultado_80['importancia'].cumsum()
print(resultado_80.to_string(index=False, formatters={
    'importancia': '{:.4f}'.format,
    'importancia_acumulada': '{:.4f}'.format,
}))

  variable importancia importancia_acumulada
enginesize      0.6748                0.6748
curbweight      0.2265                0.9013


In [51]:
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

# Usar las variables cuya importancia acumulada alcanza al menos el 80%.
X = df[variables_80.index.tolist()]
y = df['price']

X_train_80, X_test_80, y_train_80, y_test_80 = train_test_split(
    X, y, test_size=0.2, random_state=0
)

modelo_arbol_80 = DecisionTreeRegressor(max_depth=5, random_state=0)
modelo_arbol_80.fit(X_train_80, y_train_80)

r2_train_80 = r2_score(y_train_80, modelo_arbol_80.predict(X_train_80))
r2_test_80 = r2_score(y_test_80, modelo_arbol_80.predict(X_test_80))
print(f'R2_train: {r2_train_80:.4f}')
print(f'R2_test: {r2_test_80:.4f}')

R2_train: 0.9496
R2_test: 0.9145


In [52]:
print(f"X_train.shape = {X_train.shape}")
print(f"X_test.shape = {X_test.shape}")
print(f"y_train.shape = {y_train.shape}")
print(f"y_test.shape = {y_test.shape}")

X_train.shape = (164, 173)
X_test.shape = (41, 173)
y_train.shape = (164,)
y_test.shape = (41,)


In [53]:
diferencia = r2_train_80 - r2_test_80
print(round(diferencia, 4))

0.0351


In [54]:
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

y_pred_train_80 = modelo_arbol_80.predict(X_train_80)
y_pred_test_80 = modelo_arbol_80.predict(X_test_80)

mape_train_80 = mean_absolute_percentage_error(y_train_80, y_pred_train_80) * 100
mape_test_80 = mean_absolute_percentage_error(y_test_80, y_pred_test_80) * 100
rmse_train_80 = mean_squared_error(y_train_80, y_pred_train_80) ** 0.5
rmse_test_80 = mean_squared_error(y_test_80, y_pred_test_80) ** 0.5

print(f'MAPE_train = {mape_train_80:.2f}%')
print(f'MAPE_test = {mape_test_80:.2f}%')
print(f'RMSE_train = {rmse_train_80:.2f}')
print(f'RMSE_test = {rmse_test_80:.2f}')

MAPE_train = 9.39%
MAPE_test = 15.71%
RMSE_train = 1739.25
RMSE_test = 2573.38


# Caso 2

In [55]:
# Funciones auxiliares — Caso 2: Regresión logística (Phone Classification)
# Son las mismas que usamos en los laboratorios de regresión logística y árboles.
# Ejecuta esta celda una vez; después sólo llama a las funciones.
#   evaluar_clasificador y plot_matriz_confusion se ajustaron para funcionar
#      con MÁS DE DOS clases (en Titanic sólo había dos: sobrevivió / no sobrevivió).
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.preprocessing import StandardScaler


def plot_frequencies(df, columns, top_n=None):
    """Gráfica de barras con la frecuencia de cada categoría (top_n limita a las más comunes)."""
    for col in columns:
        freq = df[col].value_counts()
        if top_n:
            freq = freq.head(top_n)
        freq_df = freq.rename_axis(col).reset_index(name='Frecuencia')

        title = f'Frecuencias de {col}'
        if top_n and df[col].nunique() > top_n:
            title += f' (top {top_n})'

        fig = px.bar(freq_df, x=col, y='Frecuencia', title=title)
        fig.update_layout(xaxis={'categoryorder': 'total descending'})
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def escalar(X_train, X_test):
    """Estandariza (media 0, desviación 1) ajustando sólo con train. Regresa ambos DataFrames."""
    escalador = StandardScaler().fit(X_train)
    X_train_s = pd.DataFrame(escalador.transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test_s = pd.DataFrame(escalador.transform(X_test), columns=X_test.columns, index=X_test.index)
    return X_train_s, X_test_s


def plot_rfecv_clasificacion(rfecv, metrica='exactitud'):
    """Desempeño promedio de validación cruzada según el número de variables que conserva RFECV."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=rfecv.cv_results_['n_features'],
        y=rfecv.cv_results_['mean_test_score'],
        mode='lines+markers',
        line=dict(color='steelblue', width=3),
        marker=dict(size=7),
        name=f'{metrica} promedio (CV)'
    ))
    fig.add_vline(x=rfecv.n_features_, line_dash='dash', line_color='red',
                  annotation_text=f'elegido: {rfecv.n_features_}')
    fig.update_layout(
        title='RFECV — desempeño según número de variables seleccionadas',
        xaxis_title='Número de variables',
        yaxis_title=f'{metrica.capitalize()} (validación cruzada)',
        template='plotly_white', width=900, height=450
    )
    fig.show()


def evaluar_clasificador(nombre, y_true, y_pred):
    """Imprime y regresa exactitud, precisión, recall y F1.

    Con más de dos clases, precisión, recall y F1 son el promedio simple
    de las clases (average='macro').
    """
    binario = len(np.unique(np.concatenate([np.asarray(y_true), np.asarray(y_pred)]))) <= 2
    promedio = 'binary' if binario else 'macro'
    metricas = {
        'modelo': nombre,
        'exactitud': round(accuracy_score(y_true, y_pred), 4),
        'precisión': round(precision_score(y_true, y_pred, average=promedio, zero_division=0), 4),
        'recall': round(recall_score(y_true, y_pred, average=promedio, zero_division=0), 4),
        'F1': round(f1_score(y_true, y_pred, average=promedio, zero_division=0), 4),
    }
    print(f"{nombre}: exactitud {metricas['exactitud']} | precisión {metricas['precisión']} "
          f"| recall {metricas['recall']} | F1 {metricas['F1']}")
    return metricas


def plot_matriz_confusion(y_true, y_pred, clases=None, etiquetas=None, title='Matriz de confusión'):
    """Aciertos y errores del clasificador: la diagonal son los aciertos.

    clases: los valores de la variable objetivo en el orden en que quieres verlos,
            p. ej. ['Low', 'Mid_Low', 'Mid', 'High']. Si no las pasas, se ordenan alfabéticamente.
    etiquetas: nombres a mostrar para esas clases (opcional; por defecto, las mismas clases).
    """
    if clases is None:
        clases = sorted(set(np.asarray(y_true)) | set(np.asarray(y_pred)))
    clases = list(clases)
    etiquetas = list(etiquetas) if etiquetas is not None else clases

    cm = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=clases),
        index=[f'Real: {e}' for e in etiquetas],
        columns=[f'Predicho: {e}' for e in etiquetas]
    )

    fig = px.imshow(cm, text_auto=True, color_continuous_scale='Blues', title=title)
    fig.update_layout(width=650, height=450)
    fig.show()
    return cm

In [56]:
import pandas as pd

url = "https://raw.githubusercontent.com/jorgermzg15/data-files/refs/heads/main/phone_classification.csv"
df = pd.read_csv(url)
df.head()

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,px_height,px_width,ram,talk_time,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,20,756,2549,19,0,1,Mid_Low
1,1021,1,0.5,1,0,1,53,0.7,136,3,905,1988,2631,7,1,0,Mid
2,563,1,0.5,1,2,1,41,0.9,145,5,1263,1716,2603,9,1,0,Mid
3,615,1,2.5,0,0,0,10,0.8,131,6,1216,1786,2769,11,0,0,Mid
4,1821,1,1.2,0,13,1,44,0.6,141,2,1208,1212,1411,15,1,0,Mid_Low
